In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/finla-dataset/FINAL_TRAINING_DATASET_20251225_110326.csv


In [11]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

# Load data
df = pd.read_csv('/kaggle/input/finla-dataset/FINAL_TRAINING_DATASET_20251225_110326.csv')
df = df.dropna(subset=['training_text', 'label'])
df['training_text'] = df['training_text'].astype(str)

# Encode labels
if df['label'].dtype == 'object':
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['training_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Tokenize
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

# Dataset class
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ToxicDataset(train_encodings, train_labels)
val_dataset = ToxicDataset(val_encodings, val_labels)

print(f"✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}")

✅ Train: 4247, Val: 1062


In [9]:
# Cài đặt phiên bản protobuf ổn định cho Transformers
!pip install "protobuf<4.21" --no-deps --force-reinstall
# Đảm bảo thư viện accelerate được cập nhật để chạy Trainer mượt hơn
!pip install accelerate -U -q

  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3


In [14]:
# ============================================================================
# 0. ÉP KAGGLE NHẬN 2 GPU
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# ============================================================================
# 1. IMPORT
# ============================================================================
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch
import torch.nn as nn
import numpy as np

# ============================================================================
# 2. KHỞI TẠO MÔ HÌNH PhoBERT v2
# ============================================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=3
)

# ============================================================================
# 3. CLASS WEIGHTS
# ============================================================================
class_weights = torch.tensor(
    [0.80071644, 1.01700192, 1.30236124],
    dtype=torch.float32
)

print(f"✅ Class weights: {class_weights}")

# ============================================================================
# 4. CUSTOM TRAINER - FIX: THÊM num_items_in_batch
# ============================================================================
class SupremeTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        🔴 FIX: Multi-GPU DataParallel compatible
        """
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        # FIX: Lấy num_labels từ model wrapper-safe
        # Nếu DataParallel → model.module.config
        # Nếu single GPU → model.config
        if hasattr(model, 'module'):
            num_labels = model.module.config.num_labels
        else:
            num_labels = model.config.num_labels

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ============================================================================
# 5. METRICS
# ============================================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# ============================================================================
# 6. TRAINING ARGUMENTS
# ============================================================================
training_args = TrainingArguments(
    output_dir="./phobert_supreme_model",

    num_train_epochs=10,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    fp16=True,
    dataloader_num_workers=4,
    save_total_limit=2,

    logging_steps=50,
    report_to="none",
    seed=42
)

# ============================================================================
# 7. KIỂM TRA TRƯỚC KHI TRAIN
# ============================================================================
print("\n" + "="*80)
print("📊 DATASET INFO")
print("="*80)
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

sample = train_dataset[0]
print(f"\n✅ Sample check:")
print(f"  Labels dtype: {sample['labels'].dtype}")
print(f"  Labels value: {sample['labels'].item()}")

print(f"\n🖥️ GPU available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

# ============================================================================
# 8. TRAINER + EARLY STOPPING
# ============================================================================
trainer = SupremeTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ============================================================================
# 9. TRAIN
# ============================================================================
print("\n" + "="*80)
print("🚀 BẮT ĐẦU HUẤN LUYỆN PhoBERT v2 với GPU T4 ×2")
print("="*80)

try:
    trainer.train()
    print("\n✅ Huấn luyện hoàn thành!")
except Exception as e:
    print(f"\n❌ LỖI: {e}")
    import traceback
    traceback.print_exc()
    raise

# ============================================================================
# 10. ĐÁNH GIÁ CUỐI
# ============================================================================
print("\n" + "="*80)
print("📊 KẾT QUẢ CUỐI CÙNG")
print("="*80)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\n" + "="*80)
print("📋 CLASSIFICATION REPORT")
print("="*80)

predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=["Sạch (0)", "Công kích (1)", "Thù ghét (2)"],
    digits=4
))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print("\n🔢 Confusion Matrix:")
print(cm)

# ============================================================================
# 11. LƯU MODEL
# ============================================================================
print("\n" + "="*80)
print("💾 LƯU MODEL")
print("="*80)

trainer.save_model("./phobert_toxic_final")
print("✅ Model đã lưu tại: ./phobert_toxic_final")

print("\n" + "="*80)
print("🎉 HOÀN TẤT!")
print("="*80)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Class weights: tensor([0.8007, 1.0170, 1.3024])

📊 DATASET INFO
Train size: 4247
Val size: 1062

✅ Sample check:
  Labels dtype: torch.int64
  Labels value: 0

🖥️ GPU available: True
GPU count: 2

🚀 BẮT ĐẦU HUẤN LUYỆN PhoBERT v2 với GPU T4 ×2


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.072700,0.945506,0.531073,0.530804,0.523356
2,0.944600,0.848660,0.635593,0.627568,0.631817
3,0.745100,0.769609,0.675141,0.676980,0.675596
4,0.664600,0.759105,0.666667,0.668096,0.667661
5,0.588200,0.770254,0.685499,0.687296,0.684717
6,0.472400,0.794092,0.697740,0.698692,0.698371
7,0.397800,0.824277,0.681733,0.683197,0.683074
8,0.383800,0.834256,0.693974,0.693473,0.694109
9,0.330100,0.844327,0.688324,0.688713,0.689202



✅ Huấn luyện hoàn thành!

📊 KẾT QUẢ CUỐI CÙNG


  eval_loss: 0.7941
  eval_accuracy: 0.6977
  eval_f1_macro: 0.6987
  eval_f1_weighted: 0.6984
  eval_runtime: 8.7350
  eval_samples_per_second: 121.5790
  eval_steps_per_second: 3.8920
  epoch: 9.0000

📋 CLASSIFICATION REPORT
               precision    recall  f1-score   support

     Sạch (0)     0.7475    0.6900    0.7176       442
Công kích (1)     0.6324    0.6724    0.6518       348
 Thù ghét (2)     0.7113    0.7426    0.7266       272

     accuracy                         0.6977      1062
    macro avg     0.6971    0.7017    0.6987      1062
 weighted avg     0.7005    0.6977    0.6984      1062


🔢 Confusion Matrix:
[[305  96  41]
 [ 73 234  41]
 [ 30  40 202]]

💾 LƯU MODEL
✅ Model đã lưu tại: ./phobert_toxic_final

🎉 HOÀN TẤT!


In [21]:
!nvidia-smi


Thu Dec 25 08:38:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             32W /   70W |    5909MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

# Load data
df = pd.read_csv('/kaggle/input/dataset-toxic-gold-samples/FINAL_TRAINING_DATASET_TEENCODE_20251225_151716.csv')
df = df.dropna(subset=['training_text', 'label'])
df['training_text'] = df['training_text'].astype(str)

# Encode labels
if df['label'].dtype == 'object':
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['training_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Tokenize
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

# Dataset class
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ToxicDataset(train_encodings, train_labels)
val_dataset = ToxicDataset(val_encodings, val_labels)

print(f"✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}")

✅ Train: 4233, Val: 1059


In [23]:
# ============================================================================
# 0. ÉP KAGGLE NHẬN 2 GPU
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# ============================================================================
# 1. IMPORT
# ============================================================================
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch
import torch.nn as nn
import numpy as np

# ============================================================================
# 2. KHỞI TẠO MÔ HÌNH PhoBERT v2
# ============================================================================
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=3
)

# ============================================================================
# 3. CLASS WEIGHTS
# ============================================================================
class_weights = torch.tensor(
    [0.80071644, 1.01700192, 1.30236124],
    dtype=torch.float32
)

print(f"✅ Class weights: {class_weights}")

# ============================================================================
# 4. CUSTOM TRAINER - FIX: THÊM num_items_in_batch
# ============================================================================
class SupremeTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        🔴 FIX: Multi-GPU DataParallel compatible
        """
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        # FIX: Lấy num_labels từ model wrapper-safe
        # Nếu DataParallel → model.module.config
        # Nếu single GPU → model.config
        if hasattr(model, 'module'):
            num_labels = model.module.config.num_labels
        else:
            num_labels = model.config.num_labels

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ============================================================================
# 5. METRICS
# ============================================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# ============================================================================
# 6. TRAINING ARGUMENTS
# ============================================================================
training_args = TrainingArguments(
    output_dir="./phobert_supreme_model",

    num_train_epochs=10,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    fp16=True,
    dataloader_num_workers=4,
    save_total_limit=2,

    logging_steps=50,
    report_to="none",
    seed=42
)

# ============================================================================
# 7. KIỂM TRA TRƯỚC KHI TRAIN
# ============================================================================
print("\n" + "="*80)
print("📊 DATASET INFO")
print("="*80)
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

sample = train_dataset[0]
print(f"\n✅ Sample check:")
print(f"  Labels dtype: {sample['labels'].dtype}")
print(f"  Labels value: {sample['labels'].item()}")

print(f"\n🖥️ GPU available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

# ============================================================================
# 8. TRAINER + EARLY STOPPING
# ============================================================================
trainer = SupremeTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ============================================================================
# 9. TRAIN
# ============================================================================
print("\n" + "="*80)
print("🚀 BẮT ĐẦU HUẤN LUYỆN PhoBERT v2 với GPU T4 ×2")
print("="*80)

try:
    trainer.train()
    print("\n✅ Huấn luyện hoàn thành!")
except Exception as e:
    print(f"\n❌ LỖI: {e}")
    import traceback
    traceback.print_exc()
    raise

# ============================================================================
# 10. ĐÁNH GIÁ CUỐI
# ============================================================================
print("\n" + "="*80)
print("📊 KẾT QUẢ CUỐI CÙNG")
print("="*80)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\n" + "="*80)
print("📋 CLASSIFICATION REPORT")
print("="*80)

predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=["Sạch (0)", "Công kích (1)", "Thù ghét (2)"],
    digits=4
))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print("\n🔢 Confusion Matrix:")
print(cm)

# ============================================================================
# 11. LƯU MODEL
# ============================================================================
print("\n" + "="*80)
print("💾 LƯU MODEL")
print("="*80)

trainer.save_model("./phobert_toxic_V2_final")
print("✅ Model đã lưu tại: ./phobert_toxic_final")

print("\n" + "="*80)
print("🎉 HOÀN TẤT!")
print("="*80)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Class weights: tensor([0.8007, 1.0170, 1.3024])

📊 DATASET INFO
Train size: 4233
Val size: 1059

✅ Sample check:
  Labels dtype: torch.int64
  Labels value: 2

🖥️ GPU available: True
GPU count: 2

🚀 BẮT ĐẦU HUẤN LUYỆN PhoBERT v2 với GPU T4 ×2


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.079900,0.945415,0.536355,0.532954,0.522623
2,0.943900,0.786916,0.668555,0.659287,0.666232
3,0.755100,0.728012,0.677054,0.675970,0.677478
4,0.664900,0.712065,0.697828,0.695978,0.697818
5,0.588900,0.719862,0.711048,0.710220,0.711351
6,0.479400,0.755862,0.708215,0.707034,0.709100
7,0.429900,0.771270,0.706327,0.704189,0.706268
8,0.392300,0.792134,0.709160,0.708893,0.709857



✅ Huấn luyện hoàn thành!

📊 KẾT QUẢ CUỐI CÙNG


  eval_loss: 0.7199
  eval_accuracy: 0.7110
  eval_f1_macro: 0.7102
  eval_f1_weighted: 0.7114
  eval_runtime: 9.4805
  eval_samples_per_second: 111.7030
  eval_steps_per_second: 3.5860
  epoch: 8.0000

📋 CLASSIFICATION REPORT
               precision    recall  f1-score   support

     Sạch (0)     0.7741    0.6854    0.7271       445
Công kích (1)     0.6598    0.6881    0.6737       327
 Thù ghét (2)     0.6883    0.7770    0.7300       287

     accuracy                         0.7110      1059
    macro avg     0.7074    0.7168    0.7102      1059
 weighted avg     0.7156    0.7110    0.7114      1059


🔢 Confusion Matrix:
[[305  88  52]
 [ 53 225  49]
 [ 36  28 223]]

💾 LƯU MODEL
✅ Model đã lưu tại: ./phobert_toxic_final

🎉 HOÀN TẤT!


In [24]:
import pandas as pd
import numpy as np

# 1. Lấy dự đoán từ validation set
print("🔍 Đang trích xuất các mẫu đoán sai...")
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# 2. Tạo mapping nhãn để đọc cho dễ
label_map = {0: "Sạch (0)", 1: "Công kích (1)", 2: "Thù ghét (2)"}

# 3. Tạo DataFrame kết quả
error_df = pd.DataFrame({
    'Text': val_texts,
    'True_Label_ID': y_true,
    'Pred_Label_ID': y_pred,
    'True_Label': [label_map[i] for i in y_true],
    'Pred_Label': [label_map[i] for i in y_pred]
})

# 4. Lọc ra các câu đoán sai
misclassified = error_df[error_df['True_Label_ID'] != error_df['Pred_Label_ID']]

# 5. Phân loại các nhóm lỗi phổ biến để "soi" cho nhanh
# Lỗi nhầm Sạch -> Công kích (Thường là do Positive Slang)
slang_errors = misclassified[(misclassified['True_Label_ID'] == 0) & (misclassified['Pred_Label_ID'] == 1)]

# Lỗi nhầm Sạch -> Thù ghét (Thường là do Narrative Fact/Tường thuật)
narrative_errors = misclassified[(misclassified['True_Label_ID'] == 0) & (misclassified['Pred_Label_ID'] == 2)]

# 6. Xuất ra CSV để nhóm cùng soi trên Excel
misclassified.to_csv('misclassified_samples.csv', index=False, encoding='utf-8-sig')

print(f"✅ Đã tìm thấy {len(misclassified)} mẫu đoán sai.")
print(f"💾 File đã lưu tại: misclassified_samples.csv")

# 7. Hiển thị thử 10 câu đoán sai ngẫu nhiên
print("\n📋 MỘT SỐ MẪU ĐOÁN SAI ĐIỂN HÌNH:")
print("="*80)
pd.set_option('display.max_colwidth', 150)
print(misclassified.sample(min(10, len(misclassified)))[['Text', 'True_Label', 'Pred_Label']])

🔍 Đang trích xuất các mẫu đoán sai...


✅ Đã tìm thấy 306 mẫu đoán sai.
💾 File đã lưu tại: misclassified_samples.csv

📋 MỘT SỐ MẪU ĐOÁN SAI ĐIỂN HÌNH:
                                                                                                                                                      Text  \
660  cay đắng cảnh <person> bắt quả tang vợ lén lút ngoại tình. mặc cho quỳ gối van xin hết lời nhưng <person> nhất quyết không về </s> về nó đóng thùn...   
575  vậy là chúng ta đã kết thúc chuỗi ngày tự do yêu nhau chuyển sang cuộc sống hôn nhân <emo_pos> lgbt </s> các bạn nhìn thế nào, còn tôi nhìn thì bu...   
449                                       hai thanh niên đại học là lối chửi bới cựu chiến binh </s> <person> đúng là cha làm <person> con thì đốt sách mà   
723              bắc kỳ có nghĩa là gì ? </s> cách để để thông não cho bọn phân biệt vùng miền cách thứ 1: thu điện thoại cách thứ 2: cắt mạng wifi nhà nó   
258                                                          hai thanh niên đại học là lối chửi bới